# Which frame to replay at a fixed replay slot

**Complete** · queue · ETA 0h 00m 00s 

Replay still occurs every fourth update, with batch size 1 and LR 0.000075. Each sampled slot compares the saved uniform draw with highest-loss, most-similar, and least-similar choices from the same 16-frame past-only shortlist. Subsequent training follows the same recorded periodic schedule.

| Seed | State | Audited slots | Stream frame | Candidate scoring seconds |
| --- | --- | --- | --- | --- |
 | 2026 | complete | 96/96 | 80,000 | 0.96 | 
 | 2027 | complete | 96/96 | 80,000 | 0.93 | 
 | 2028 | complete | 96/96 | 80,000 | 0.96 | 

## Reconstruction by fixed candidate rule

| Rule | Earlier clips | Later draw A | Later draw B | Older anchor B |
| --- | --- | --- | --- | --- |
 | Uniform | 0.822518 | 0.784670 | 0.785747 | 0.855224 | 
 | Highest reconstruction loss | 0.822575 | 0.784629 | 0.785648 | 0.854884 | 
 | Most similar | 0.822657 | 0.784511 | 0.785608 | 0.855169 | 
 | Least similar | 0.822680 | 0.784703 | 0.785811 | 0.855313 | 

Best fixed rule chosen on earlier clips: **Uniform**.

| Later clips | MSE |
| --- | --- |
 | Same-draw oracle gain | +0.000802 | 
 | Cross-draw oracle gain | +0.000597 | 
 | Cross-draw gain vs uniform | +0.000597 | 
 | Cross-draw lower 90% bound | +0.000480 | 
 | Older-anchor excess MSE | -0.000047 | 
 | Older-anchor upper 90% bound | +0.000009 | 

The draw-A winner repeated on **67.4%** of states under draw B. The oracle chose a different image than the fixed rule 113 times. The gain and retention gate **did not pass**. It requires at least 0.001 cross-draw gain, a positive one-sided 90% lower bound, and at most 0.001 older-anchor excess at the upper bound. Intervals resample clips.

| Group | Cross-draw gain |
| --- | --- |
 | 2026 | +0.000557 | 
 | 2027 | +0.000692 | 
 | 2028 | +0.000542 | 
 | within clip | +0.000458 | 
 | across boundary | +0.000736 | 

In [ ]:
# Run this cell to watch the worker. Interrupt the cell to stop watching.
from IPython.display import clear_output, display, Markdown
from pathlib import Path
import json, time

status_file = Path('C:\\CAFL-E\\analysis\\2026-11-09-one-step-mpc\\replay-candidate-audit\\status.json')
while True:
    try:
        status = json.loads(status_file.read_text(encoding='utf-8'))
    except PermissionError:
        time.sleep(1)
        continue
    clear_output(wait=True)
    active = status['active']
    current = next((r for r in status['queue'] if f"{r['seed']}-{r['arm']}" == active), None)
    eta = status['eta_seconds']
    display(Markdown(f"**{status['state']}** · {active or 'queue'} · ETA {eta / 3600:.2f} h" if eta is not None else f"**{status['state']}** · {active or 'queue'}"))
    if current:
        display(current)
    if status['state'] in ('complete', 'stopped', 'failed'):
        break
    time.sleep(10)
